# Eye-closure and yawning fusion model training


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

# Change PROJECT_DIR only if your repository is somewhere else.
CANDIDATE_FOLDERS = [
    Path.cwd(),
    Path('F:/isaaclab/prototype-thesis'),
    Path('/mnt/f/isaaclab/prototype-thesis'),
]
PROJECT_DIR = next((p.resolve() for p in CANDIDATE_FOLDERS
                    if (p / 'scripts/train_fusion.py').is_file() and (p / 'dms').is_dir()), None)
assert PROJECT_DIR is not None, (
    'Repository not found. Edit CANDIDATE_FOLDERS above to point to the folder containing scripts/ and dms/.')
os.chdir(PROJECT_DIR)

FUSION_TABLE = PROJECT_DIR / 'datasets/fusion/esca_phase2_train_val.csv'
FUSION_MODEL = PROJECT_DIR / 'models/esca_fusion_phase2.json'
print('REPOSITORY:', PROJECT_DIR)
print('NOTEBOOK PYTHON:', sys.executable)
print('FUSION INPUT:', FUSION_TABLE)


REPOSITORY: F:\isaaclab\prototype-thesis
NOTEBOOK PYTHON: f:\isaaclab\env_isaaclab\Scripts\python.exe
ENVIRONMENT: Windows
YOLO checkpoint: F:\isaaclab\prototype-thesis\runs\quick_driver_state_12000_e35\yolo11n\weights\best.pt


In [ ]:
import json
import time
from datetime import datetime

START = time.monotonic()

def log(message, level='INFO'):
    print(f'[{datetime.now():%H:%M:%S}] [+{time.monotonic()-START:,.0f}s] [{level}] {message}', flush=True)

def section(message):
    print('\n' + '=' * 76, flush=True)
    log(message, 'STEP')
    print('=' * 76, flush=True)

def run_verbose(*items, allow_failure=False):
    args = [str(x) for x in items]
    section('Running ' + ' '.join(args))
    started = time.monotonic()
    with subprocess.Popen(args, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                          errors='replace', bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
    log(f'Exit {status}; elapsed {(time.monotonic()-started)/60:.1f} minutes',
        'SUCCESS' if status == 0 else 'ERROR')
    if status and not allow_failure:
        raise RuntimeError(f'Command failed (exit code {status}). Read its output above.')
    return status

def exists_or_explain(path, purpose):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Missing {purpose}: {path}. Run dataset.ipynb or correct FUSION_TABLE.')
    return path




[19:41:46] [+0s] [STEP] Checking repository and Python

[19:41:46] [+0s] [STEP] Running git rev-parse --short HEAD
467ec50
[19:41:46] [+0s] [SUCCESS] Exit 0; elapsed 0.0 minutes
Python: 3.12.13
OpenCV: 5.0.0 Torch: 2.11.0+cu130
CUDA available: True
Repository requirements and core imports: OK


In [ ]:
%pip install -r requirements.txt


## Check the independent fusion input


In [7]:
exists_or_explain(PROJECT_DIR / 'scripts/train_fusion.py', 'fusion trainer')
exists_or_explain(FUSION_TABLE, 'fusion feature CSV')
print('Feature table:', FUSION_TABLE)


## Train fusion model

In [14]:
section('Fit fusion model, verify both output heads, and inspect report')

run_verbose(sys.executable, 'scripts/train_fusion.py',
            '--table', FUSION_TABLE, '--out', FUSION_MODEL,
            '--hidden', '16', '--seed', '0')
model_data = json.loads(exists_or_explain(FUSION_MODEL, 'trained fusion model').read_text(encoding='utf-8'))
heads = set(model_data.get('heads', {}))
assert {'eye_closed', 'yawn'} <= heads, (
    f'Fusion training is incomplete. Needed both eye_closed and yawn; got {sorted(heads)}.')
fusion_report = FUSION_MODEL.with_name(FUSION_MODEL.stem + '_report.md')
print('Fusion heads:', sorted(heads))
print('Fusion model:', FUSION_MODEL)
if fusion_report.is_file():
    print(fusion_report.read_text(encoding='utf-8')[:7500])


[20:40:45] [+3,539s] [STEP] Fit fusion model, verify both output heads, and inspect report

[20:40:45] [+3,539s] [STEP] Running f:\isaaclab\env_isaaclab\Scripts\python.exe scripts/train_fusion.py --table F:\isaaclab\prototype-thesis\datasets\fusion\esca_phase2_train_val.csv --out F:\isaaclab\prototype-thesis\models\esca_fusion_phase2.json --hidden 16 --seed 0
# Fusion model report

Table: `F:\isaaclab\prototype-thesis\datasets\fusion\esca_phase2_train_val.csv` - trained 2026-11-16T20:40:46


| Sign | Model kept | Test pictures (no/yes) | Rules F1 | Fusion F1 | Rules + fusion F1 |
|---|---|---|---|---|---|
| Eyes closed | mlp | [4620, 1591] | 1.000 | 0.998 | 0.998 |
| Yawning | mlp | [6264, 85] | 1.000 | 0.994 | 0.994 |

Rules = EAR / MAR and blendshape thresholds with population defaults (no calibration on single pictures).
Rules + fusion = the live program with the fusion model loaded (equal weights).

Model: F:\isaaclab\prototype-thesis\models\esca_fusion_phase2.json
Report: F:\isaa